# ST · Vicari (mouse brain, Visium)

Walks through `stsm_integration.preprocess.st` on one slice, then all slices.
Read-only by default: nothing is written unless `SAVE = True` in the last section.

| Step | Function |
|---|---|
| 1 Load | `load_visium` |
| 2 Coordinates → µm | `coords_to_um` |
| 3 QC | `qc_filter` (`mito_cutoff`) |
| 4 Normalise | `lognormalize` |
| 5 Contract check | `io.validate_adata` |
| 6 Leiden | `sc.pl.spatial` |
| 7 Expert Annotation (RegionLoupe) |  |
| 8 Save | `sc.write_h5ad` |

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors

from stsm_integration.io import load_config, validate_adata
from stsm_integration.preprocess import st as pst

sc.set_figure_params(dpi=80, frameon=False)

## 0 · Config and paths

In [ ]:
cfg = load_config("../../configs/vicari_mouse.yaml")
RAW = Path(cfg["paths"]["raw_st"])

# Space Ranger layout differs per source; adapt here, then move the rule into io.py
def outs_dir(st_name: str) -> Path:
    candidates = [RAW / st_name / "outs",
                  RAW / st_name / "output_data" / st_name / "outs"]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"no outs/ for {st_name}: tried {candidates}")

for s in cfg["slices"]:
    print(s["id"], "→", outs_dir(s["st"]))

## 1 · Load one slice

In [ ]:
SLICE = cfg["slices"][0]
adata = pst.load_visium(str(outs_dir(SLICE["st"])), SLICE["id"])
adata

In [ ]:
lib = next(iter(adata.uns["spatial"]))
print("library:", lib)
print("scalefactors:", adata.uns["spatial"][lib]["scalefactors"])
print("images:", list(adata.uns["spatial"][lib]["images"]))
print("counts dtype / max:", adata.layers["counts"].dtype, adata.layers["counts"].max())
print("in-tissue spots:", adata.n_obs, "| genes:", adata.n_vars)

## 2 · Coordinates → µm

In [ ]:
pst.coords_to_um(adata)
print(f"µm per full-res pixel: {adata.uns['um_per_px']:.3f}")
xy = adata.obsm["spatial"]
print("extent (µm):", np.ptp(xy, axis=0).round(0))

# sanity check: Visium centre-to-centre spacing is ~100 µm
d, _ = NearestNeighbors(n_neighbors=2).fit(xy).kneighbors(xy)
print("median NN distance (µm):", np.median(d[:, 1]).round(1))

## 3 · QC
Metrics are computed here only to plot them; `qc_filter` recomputes them.

In [ ]:
adata.var["mt"] = adata.var_names.str.lower().str.startswith(cfg["st"]["mito_prefix"])
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], layer="counts", percent_top=None, inplace=True)
cut = pst.mito_cutoff(adata.obs["pct_counts_mt"].to_numpy())
print(f"mito genes: {adata.var['mt'].sum()} | cutoff: {cut:.2f}%")

fig, ax = plt.subplots(1, 3, figsize=(13, 3))
ax[0].hist(adata.obs["total_counts"], 60); ax[0].set_title("total counts")
ax[1].hist(adata.obs["n_genes_by_counts"], 60); ax[1].set_title("genes per spot")
ax[2].hist(adata.obs["pct_counts_mt"], 60); ax[2].axvline(cut, c="r"); ax[2].set_title("% mito")
plt.tight_layout()

In [ ]:
# plotted on pixel coords + H&E (scanpy expects pixels)
tmp = adata.copy(); tmp.obsm["spatial"] = tmp.obsm["spatial_px"]
sc.pl.spatial(tmp, color=["total_counts", "pct_counts_mt"], ncols=2)

In [ ]:
adata_raw = adata.copy()
adata = pst.qc_filter(adata, mito_prefix=cfg["st"]["mito_prefix"],
                      min_counts=cfg["st"]["min_counts"], min_genes=cfg["st"]["min_genes"])
adata.uns["qc"]

In [ ]:
# Q5 · where are the removed spots, and why were they removed?
removed = ~adata_raw.obs_names.isin(adata.obs_names)
cut = adata.uns["qc"]["mito_cutoff_pct"]

reason = np.full(adata_raw.n_obs, "kept", dtype=object)
reason[removed & (adata_raw.obs["pct_counts_mt"] >= cut).to_numpy()] = "high mito"
reason[removed & (adata_raw.obs["pct_counts_mt"] < cut).to_numpy()] = "low counts/genes"

tmp = adata_raw.copy()
tmp.obsm["spatial"] = tmp.obsm["spatial_px"]       # pixels for the H&E overlay
tmp.obs["qc_status"] = pd.Categorical(reason, categories=["kept", "high mito", "low counts/genes"])
print(tmp.obs["qc_status"].value_counts())
sc.pl.spatial(tmp, color="qc_status", palette=["lightgrey", "crimson", "royalblue"])

## 4 · Normalise (no scaling)

In [ ]:
pst.lognormalize(adata)
print("X min/max:", adata.X.min(), adata.X.max())
adata.uns["modality"] = "st"
adata.uns["pixel_size_um"] = cfg["st"]["spot_diameter_um"]

## 5 · Contract check

In [ ]:
validate_adata(adata, "st")
print("contract OK")

## 6 · Quick look (not saved)
Leiden here is for eyeballing only. It is **not** a reference label.

In [ ]:
tmp = adata.copy(); tmp.obsm["spatial"] = tmp.obsm["spatial_px"]
sc.pp.highly_variable_genes(tmp, n_top_genes=2000, flavor="seurat_v3", layer="counts")
sc.pp.pca(tmp, mask_var="highly_variable")
sc.pp.neighbors(tmp)
sc.tl.leiden(tmp, resolution=0.5, flavor="igraph", n_iterations=2)
sc.pl.spatial(tmp, color="leiden")

## 7 · Expert annotations (RegionLoupe)
Vicari ships `RegionLoupe.csv` for A1/B1 in `outs/`. Attached here for P5; not used yet.

In [ ]:
ann = outs_dir(SLICE["st"]) / "RegionLoupe.csv"
if ann.exists():
    df = pd.read_csv(ann, index_col="Barcode")
    adata.obs["RegionLoupe"] = adata.obs_names.map(df.iloc[:, 0])
    print(adata.obs["RegionLoupe"].value_counts(dropna=False))
    tmp = adata.copy(); tmp.obsm["spatial"] = tmp.obsm["spatial_px"]
    sc.pl.spatial(tmp, color="RegionLoupe")
else:
    print("no RegionLoupe.csv for", SLICE["id"])

## 8 · All slices

In [ ]:
SAVE = True
cfg = load_config("../../configs/vicari_mouse.yaml")
out = Path(cfg["paths"]["processed_st"])

summary = []
for s in cfg["slices"]:
    a = pst.preprocess_st(pst.load_visium(str(outs_dir(s["st"])), s["id"]), cfg, slice_cfg=s)
    validate_adata(a, "st")
    summary.append({"slice": s["id"], **a.uns["qc"]})
    if SAVE:
        out.mkdir(parents=True, exist_ok=True)
        a.write_h5ad(out / f"{s['id']}.h5ad")

pd.DataFrame(summary)

In [ ]:
s = cfg["slices"][1]                       # B1; use [2] for C1
raw = pst.load_visium(str(outs_dir(s["st"])), s["id"])
pst.coords_to_um(raw)

# QC metrics on raw itself (same call qc_filter makes internally)
raw.var["mt"] = raw.var_names.str.lower().str.startswith(cfg["st"]["mito_prefix"])
sc.pp.calculate_qc_metrics(raw, qc_vars=["mt"], layer="counts", percent_top=None, inplace=True)

kept = pst.qc_filter(raw.copy(), mito_prefix=cfg["st"]["mito_prefix"])
q = kept.uns["qc"]
raw.obs["removed"] = (~raw.obs_names.isin(kept.obs_names)).astype(str)
print(f"{s['id']}: removed {(raw.obs['removed'] == 'True').sum()} / {raw.n_obs} "
      f"| min_counts {q['min_counts']:.0f} | min_genes {q['min_genes']:.0f}")

ann = outs_dir(s["st"]) / "RegionLoupe.csv"
if ann.exists():
    raw.obs["region"] = raw.obs_names.map(pd.read_csv(ann, index_col="Barcode").iloc[:, 0])
    print(pd.crosstab(raw.obs["region"], raw.obs["removed"], normalize="index").round(2))

fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].hist(np.log1p(raw.obs["total_counts"]), 60)
ax[0].axvline(np.log1p(kept.uns["qc"]["min_counts"]), c="r"); ax[0].set_title("log1p total counts")
ax[1].hist(np.log1p(raw.obs["n_genes_by_counts"]), 60)
ax[1].axvline(np.log1p(kept.uns["qc"]["min_genes"]), c="r"); ax[1].set_title("log1p genes")
tmp = raw.copy(); tmp.obsm["spatial"] = tmp.obsm["spatial_px"]
sc.pl.spatial(tmp, color="removed", palette=["lightgrey", "royalblue"])

In [ ]:
s = cfg["slices"][2]                       # B1; use [2] for C1
raw = pst.load_visium(str(outs_dir(s["st"])), s["id"])
pst.coords_to_um(raw)

# QC metrics on raw itself (same call qc_filter makes internally)
raw.var["mt"] = raw.var_names.str.lower().str.startswith(cfg["st"]["mito_prefix"])
sc.pp.calculate_qc_metrics(raw, qc_vars=["mt"], layer="counts", percent_top=None, inplace=True)

kept = pst.qc_filter(raw.copy(), mito_prefix=cfg["st"]["mito_prefix"])
q = kept.uns["qc"]
raw.obs["removed"] = (~raw.obs_names.isin(kept.obs_names)).astype(str)
print(f"{s['id']}: removed {(raw.obs['removed'] == 'True').sum()} / {raw.n_obs} "
      f"| min_counts {q['min_counts']:.0f} | min_genes {q['min_genes']:.0f}")

ann = outs_dir(s["st"]) / "RegionLoupe.csv"
if ann.exists():
    raw.obs["region"] = raw.obs_names.map(pd.read_csv(ann, index_col="Barcode").iloc[:, 0])
    print(pd.crosstab(raw.obs["region"], raw.obs["removed"], normalize="index").round(2))

fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].hist(np.log1p(raw.obs["total_counts"]), 60)
ax[0].axvline(np.log1p(kept.uns["qc"]["min_counts"]), c="r"); ax[0].set_title("log1p total counts")
ax[1].hist(np.log1p(raw.obs["n_genes_by_counts"]), 60)
ax[1].axvline(np.log1p(kept.uns["qc"]["min_genes"]), c="r"); ax[1].set_title("log1p genes")
tmp = raw.copy(); tmp.obsm["spatial"] = tmp.obsm["spatial_px"]
sc.pl.spatial(tmp, color="removed", palette=["lightgrey", "royalblue"])